# Packages Import

In [36]:
import os
import json
import requests
from bs4 import BeautifulSoup

# Ceneo Scraper

1. Provide url of the product's opinions page

In [9]:
product_code = "124893467"
page = 1
url = f"https://www.ceneo.pl/{product_code}/opinie-{page}"

2. Send request to provided  url

In [10]:
response = requests.get(url)
print(response.status_code)

200


3. Fetch product name

In [11]:
page_dom = BeautifulSoup(response.text, 'html.parser')
print(type(page_dom))

<class 'bs4.BeautifulSoup'>


In [12]:
product_name = page_dom.select_one('h1').get_text()
print(product_name)
print(type(product_name))

Urządzenie wielofunkcyjne HP Smart Tank 670 AiO (6UU48A)
<class 'str'>


4. Fetchall opinions from the webpage

In [13]:
opinions = page_dom.select("div.js_product-review:not(.user-post--highlight)")
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
10


5. Parse opinion to extract required data

In [21]:
OPINION_CONTAINER = "div.js_product-review:not(.user-post--highlight)"

def get_text_safe(el):
    return el.get_text(strip=True) if el else None

def get_attr_safe(el, attr):
    return el.get(attr) if el else None

SELECTORS = {
    "opinion_id": lambda op: op.get("data-entry-id"),

    "author_recommendation": lambda op: get_text_safe(
        op.select_one("span.user-post__author-recomendation > em")
    ),

    "score_stars": lambda op: get_text_safe(
        op.select_one("span.user-post__score-count")
    ),

    "content": lambda op: get_text_safe(
        op.select_one("div.user-post__text")
    ),

    "pros": lambda op: [
        el.get_text(strip=True)
        for el in op.select("div.review-feature__item--positive")
    ],

    "cons": lambda op: [
        el.get_text(strip=True)
        for el in op.select("div.review-feature__item--negative")
    ],

    "helpful_count": lambda op: get_attr_safe(
        op.select_one("button.vote-yes"), "data-total-vote"
    ),

    "unhelpful_count": lambda op: get_attr_safe(
        op.select_one("button.vote-no"), "data-total-vote"
    ),

    "publishing_date": lambda op: get_text_safe(
        op.select_one("span.user-post__published > time:nth-of-type(1)")
    ),

    "purchase_date": lambda op: get_text_safe(
        op.select_one("span.user-post__published > time:nth-of-type(2)")
    ),
}

opinions = page_dom.select(OPINION_CONTAINER)

for op in opinions:
    data = {key: func(op) for key, func in SELECTORS.items()}
    print(data)


{'opinion_id': '17657913', 'author_recommendation': 'Polecam', 'score_stars': '5/5', 'content': '#Promocja-HP\nWszystko w jak najlepszym porządeczku. Drukarka jest bardzo ekonomiczna, posiada druk dwustronny, oraz opcję drukowania prosto że smartfona. Bardzo przydatną funkcją są umieszczone na froncie wskaźniki poziomu poszczególnych tuszy. Do użytku domowego drukarka jest po prostu idealna. Polecam.', 'pros': [], 'cons': [], 'helpful_count': '0', 'unhelpful_count': '0', 'publishing_date': '3 lata temu,', 'purchase_date': 'po 3 tygodniach'}
{'opinion_id': '19466347', 'author_recommendation': 'Polecam', 'score_stars': '5/5', 'content': 'To moja pierwsza drukarka atramentowa od lat. Łatwość instalacji wg instrukcji i jakość druku dokumentów obłędne! Jestem zachwycony jakością. Bardzo wygodne drukowanie z dowolnego miejsca i urządzenia po podłączeniu do konta HP Smart. Dokładnie to, czego oczekiwałem. Pełne zadowolenie.', 'pros': ['czyste napełnianie atramentem', 'jakość wydruków', 'niski

In [24]:
all_opinions = []
for opinion in opinions:
    single_opinion = {
        'opinion_id': opinion.get("data-entry-id"),
        'author': opinion.select_one("span.user-post__author-name").get_text().strip(),
        'recommendation': opinion.select_one("span.user-post__author-recomendation > em").get_text().strip() if opinion.select_one("span.user-post__author-recomendation > em") else None,
        'score': opinion.select_one("span.user-post__score-count").get_text().strip(),
        'content': opinion.select_one("div.user-post__text").get_text().strip(),
        'pros': [p.get_text().strip() for p in opinion.select("div.review-feature__item--positive")],
        'cons': [c.get_text().strip() for c in opinion.select("div.review-feature__item--negative")],
        'helpful': opinion.select_one("button.vote-yes > span").get_text().strip(),
        'unhelpful': opinion.select_one("button.vote-no > span").get_text().strip(),
        'publish_date': opinion.select_one("span.user-post__published > time:nth-child(1)[datetime]").get('datetime').strip(),
        'purchase_date': opinion.select_one("span.user-post__published > time:nth-child(2)[datetime]").get('datetime').strip() if opinion.select_one("span.user-post__published > time:nth-child(2)[datetime]")  else None

    }
    all_opinions.append(single_opinion)

In [25]:
print(all_opinions)

[{'opinion_id': '17657913', 'author': 'l...z', 'recommendation': 'Polecam', 'score': '5/5', 'content': '#Promocja-HP\nWszystko w jak najlepszym porządeczku. Drukarka jest bardzo ekonomiczna, posiada druk dwustronny, oraz opcję drukowania prosto że smartfona. Bardzo przydatną funkcją są umieszczone na froncie wskaźniki poziomu poszczególnych tuszy. Do użytku domowego drukarka jest po prostu idealna. Polecam.', 'pros': [], 'cons': [], 'helpful': '0', 'unhelpful': '0', 'publish_date': '2023-06-28 22:50:27', 'purchase_date': '2023-06-12 10:14:50'}, {'opinion_id': '19466347', 'author': 'Tomasz', 'recommendation': 'Polecam', 'score': '5/5', 'content': 'To moja pierwsza drukarka atramentowa od lat. Łatwość instalacji wg instrukcji i jakość druku dokumentów obłędne! Jestem zachwycony jakością. Bardzo wygodne drukowanie z dowolnego miejsca i urządzenia po podłączeniu do konta HP Smart. Dokładnie to, czego oczekiwałem. Pełne zadowolenie.', 'pros': ['czyste napełnianie atramentem', 'jakość wydruk

6. Check if there is next page with opinions

In [26]:
next = True if page_dom.select_one('button.pagination_next') else False
if next: page += 1 

8. Save acquired opnions

In [33]:
if not os.path.exists("./opinions"):
    os.mkdir("./opinions")

In [37]:
with open(f"./opinions/{product_code}.json", "w", encoding="UTF-8") as jf:
    json.dump(all_opinions, jf, indent=4, ensure_ascii= False)